In [1]:
import os
import pickle
import joblib
import math
import csv
import numpy as np
import pandas as pd
from collections import defaultdict
from itertools import combinations
import networkx as nx
from PIL import Image


In [ ]:
ROOM_CODE = {
    'common_room': 1,
    'master_room': 2,
    'living_room': 3,
    'balcony': 4,
    'bathroom': 5,
    'kitchen': 6,
    'storage': 7,
    'dining': 8,
}

In [28]:
meta_df = pd.read_csv("./output/reorientation_metadata.csv", index_col="filename")

list_of_plan_dicts = pickle.load(open("./output/graphs_reoriented.pkl", "rb"))

In [13]:
plan = list_of_plan_dicts[0]
row = meta_df.loc[plan['filename']]
cx, cy = row["center_x"], row["center_y"]
# (rotation_angle is available as row["rotation_angle"] if you need it)

# --- recenter & flip Y so that (cx,cy) → (0,0) and Y points up ------------
for r in plan['rooms']:
    x, y = r['centroid']
    r['x0'] = x - cx
    r['y0'] = -(y - cy)

In [14]:
plan

{'filename': '40795.png',
 'rooms': [{'type': 'common_room',
   'polygon': <POLYGON ((69 80, 69 121, 117 121, 117 80, 69 80))>,
   'area': 1968.0,
   'centroid': (93.0, 100.5),
   'x0': -18.0,
   'y0': 60.5},
  {'type': 'common_room',
   'polygon': <POLYGON ((23 100, 23 134, 65 134, 65 100, 23 100))>,
   'area': 1428.0,
   'centroid': (44.0, 117.0),
   'x0': -67.0,
   'y0': 44.0},
  {'type': 'master_room',
   'polygon': <POLYGON ((152 80, 152 121, 214 121, 214 80, 152 80))>,
   'area': 2542.0,
   'centroid': (183.0, 100.5),
   'x0': 72.0,
   'y0': 60.5},
  {'type': 'living_room',
   'polygon': <POLYGON ((121 107, 121 124, 120 125, 69 125, 69 159, 120 159, 121 160, 121 ...>,
   'area': 6998.5,
   'centroid': (146.88495153723417, 145.95382343835584),
   'x0': 35.88495153723417,
   'y0': 15.04617656164416},
  {'type': 'balcony',
   'polygon': <POLYGON ((218 80, 218 121, 233 121, 233 80, 218 80))>,
   'area': 615.0,
   'centroid': (225.5, 100.5),
   'x0': 114.5,
   'y0': 60.5},
  {'type': 

In [21]:
uni_acc = defaultdict(lambda: {'count':0,'area':0.0,'ax':0.0,'ay':0.0})
for r in plan['rooms']:
    c = ROOM_CODE[r['type']]
    a = r['area']
    uni_acc[c]['count'] += 1
    uni_acc[c]['area']  += a
    uni_acc[c]['ax']    += a * r['x0']
    uni_acc[c]['ay']    += a * r['y0']

unigram_feats = {}
for c, d in uni_acc.items():
    total_a = d['area']

    unigram_feats[c] = {
        'count':      d['count'],
        'total_area': total_a,
        'avg_x':      d['ax'] / total_a if total_a>0 else 0.0,
        'avg_y':      d['ay' ]/ total_a if total_a>0 else 0.0,
    }


In [22]:
unigram_feats

{1: {'count': 2,
  'total_area': 3396.0,
  'avg_x': -38.60424028268551,
  'avg_y': 53.561837455830386},
 2: {'count': 1, 'total_area': 2542.0, 'avg_x': 72.0, 'avg_y': 60.5},
 3: {'count': 1,
  'total_area': 6998.5,
  'avg_x': 35.88495153723417,
  'avg_y': 15.04617656164416},
 4: {'count': 1, 'total_area': 615.0, 'avg_x': 114.5, 'avg_y': 60.5},
 5: {'count': 1, 'total_area': 621.0, 'avg_x': 23.5, 'avg_y': 69.5},
 6: {'count': 1, 'total_area': 882.0, 'avg_x': -67.0, 'avg_y': 12.5}}

In [23]:
total_area = {c: uni_acc[c]['area'] for c in uni_acc}

# --- BIGRAMS -------------------------------------------------------------
mid_acc = defaultdict(lambda: {'count':0,'sx':0.0,'sy':0.0})
G = plan['graph']
for u, v in G.edges():
    cu = ROOM_CODE[plan['rooms'][u]['type']]
    cv = ROOM_CODE[plan['rooms'][v]['type']]
    i, j = sorted((cu, cv))
    xmid = 0.5*(plan['rooms'][u]['x0'] + plan['rooms'][v]['x0'])
    ymid = 0.5*(plan['rooms'][u]['y0'] + plan['rooms'][v]['y0'])
    acc = mid_acc[(i,j)]
    acc['count'] += 1
    acc['sx']    += xmid
    acc['sy']    += ymid

bigram_feats = {}
for (i,j), d in mid_acc.items():
    cnt = d['count']
    bigram_feats[(i,j)] = {
        'count':      cnt,
        'area_ratio': total_area.get(i,0.0) / total_area.get(j,1.0),
        'avg_x':      d['sx']/cnt,
        'avg_y':      d['sy']/cnt,
    }

In [24]:
bigram_feats

{(1, 1): {'count': 1, 'area_ratio': 1.0, 'avg_x': -42.5, 'avg_y': 52.25},
 (1, 3): {'count': 2,
  'area_ratio': 0.4852468386082732,
  'avg_x': -3.3075242313829136,
  'avg_y': 33.64808828082208},
 (1, 5): {'count': 1,
  'area_ratio': 5.468599033816425,
  'avg_x': 2.75,
  'avg_y': 65.0},
 (1, 6): {'count': 1,
  'area_ratio': 3.8503401360544216,
  'avg_x': -67.0,
  'avg_y': 28.25},
 (2, 3): {'count': 1,
  'area_ratio': 0.3632206901478888,
  'avg_x': 53.942475768617086,
  'avg_y': 37.77308828082208},
 (2, 4): {'count': 1,
  'area_ratio': 4.133333333333334,
  'avg_x': 93.25,
  'avg_y': 60.5},
 (2, 5): {'count': 1,
  'area_ratio': 4.093397745571659,
  'avg_x': 47.75,
  'avg_y': 65.0},
 (3, 4): {'count': 1,
  'area_ratio': 11.379674796747967,
  'avg_x': 75.19247576861709,
  'avg_y': 37.77308828082208},
 (3, 5): {'count': 1,
  'area_ratio': 11.269726247987117,
  'avg_x': 29.692475768617086,
  'avg_y': 42.27308828082208},
 (3, 6): {'count': 1,
  'area_ratio': 7.934807256235827,
  'avg_x': -15.5

In [25]:
tri_acc = defaultdict(lambda: {'count':0,'sx':0.0,'sy':0.0})
nodes = list(range(len(plan['rooms'])))
for u, v, w in combinations(nodes, 3):
    if G.has_edge(u,v) and G.has_edge(v,w) and G.has_edge(u,w):
        cu = ROOM_CODE[plan['rooms'][u]['type']]
        cv = ROOM_CODE[plan['rooms'][v]['type']]
        cw = ROOM_CODE[plan['rooms'][w]['type']]
        i, j, k = sorted((cu, cv, cw))
        xtri = (plan['rooms'][u]['x0'] +
                plan['rooms'][v]['x0'] +
                plan['rooms'][w]['x0']) / 3.0
        ytri = (plan['rooms'][u]['y0'] +
                plan['rooms'][v]['y0'] +
                plan['rooms'][w]['y0']) / 3.0
        acc = tri_acc[(i,j,k)]
        acc['count'] += 1
        acc['sx']    += xtri
        acc['sy']    += ytri

trigram_feats = {}
for (i,j,k), d in tri_acc.items():
    cnt = d['count']
    # two ratios: i/j and j/k
    r1 = total_area.get(i,0.0) / total_area.get(j,1.0)
    r2 = total_area.get(j,0.0) / total_area.get(k,1.0)
    trigram_feats[(i,j,k)] = {
        'count':         cnt,
        'area_ratio_ij': r1,
        'area_ratio_jk': r2,
        'avg_x':         d['sx']/cnt,
        'avg_y':         d['sy']/cnt,
    }

In [26]:
trigram_feats

{(1, 1, 3): {'count': 1,
  'area_ratio_ij': 1.0,
  'area_ratio_jk': 0.4852468386082732,
  'avg_x': -16.371682820921944,
  'avg_y': 39.84872552054805},
 (1, 3, 5): {'count': 1,
  'area_ratio_ij': 0.4852468386082732,
  'area_ratio_jk': 11.269726247987117,
  'avg_x': 13.794983845744724,
  'avg_y': 48.34872552054805},
 (1, 3, 6): {'count': 1,
  'area_ratio_ij': 0.4852468386082732,
  'area_ratio_jk': 7.934807256235827,
  'avg_x': -32.705016154255276,
  'avg_y': 23.848725520548054},
 (2, 3, 4): {'count': 1,
  'area_ratio_ij': 0.3632206901478888,
  'area_ratio_jk': 11.379674796747967,
  'avg_x': 74.12831717907805,
  'avg_y': 45.34872552054805},
 (2, 3, 5): {'count': 1,
  'area_ratio_ij': 0.3632206901478888,
  'area_ratio_jk': 11.269726247987117,
  'avg_x': 43.794983845744724,
  'avg_y': 48.34872552054805}}

In [27]:
def extract_features(plan, meta_df):
    """
    plan: dict with keys
      - 'filename': str
      - 'rooms': list of dicts with keys 'type', 'centroid', 'area'
      - 'graph': networkx.Graph with nodes 0..len(rooms)-1
    meta_df: pandas DataFrame indexed by filename
    """
    # --- get center & rotation from metadata -------------------------------
    row = meta_df.loc[plan['filename']]
    cx, cy = row["center_x"], row["center_y"]
    # (rotation_angle is available as row["rotation_angle"] if you need it)

    # --- recenter & flip Y so that (cx,cy) → (0,0) and Y points up ------------
    for r in plan['rooms']:
        x, y = r['centroid']
        r['x0'] = x - cx
        r['y0'] = -(y - cy)

    # --- UNIGRAMS ------------------------------------------------------------
    uni_acc = defaultdict(lambda: {'count':0,'area':0.0,'ax':0.0,'ay':0.0})
    for r in plan['rooms']:
        c = ROOM_CODE[r['type']]
        a = r['area']
        uni_acc[c]['count'] += 1
        uni_acc[c]['area']  += a
        uni_acc[c]['ax']    += a * r['x0']
        uni_acc[c]['ay']    += a * r['y0']

    unigram_feats = {}
    for c, d in uni_acc.items():
        total_a = d['area']
        unigram_feats[c] = {
            'count':      d['count'],
            'total_area': total_a,
            'avg_x':      d['ax']/total_a if total_a>0 else 0.0,
            'avg_y':      d['ay']/total_a if total_a>0 else 0.0,
        }

    # cache total area per type for ratio calcs
    total_area = {c: uni_acc[c]['area'] for c in uni_acc}

    # --- BIGRAMS -------------------------------------------------------------
    mid_acc = defaultdict(lambda: {'count':0,'sx':0.0,'sy':0.0})
    G = plan['graph']
    for u, v in G.edges():
        cu = ROOM_CODE[plan['rooms'][u]['type']]
        cv = ROOM_CODE[plan['rooms'][v]['type']]
        i, j = sorted((cu, cv))
        xmid = 0.5*(plan['rooms'][u]['x0'] + plan['rooms'][v]['x0'])
        ymid = 0.5*(plan['rooms'][u]['y0'] + plan['rooms'][v]['y0'])
        acc = mid_acc[(i,j)]
        acc['count'] += 1
        acc['sx']    += xmid
        acc['sy']    += ymid

    bigram_feats = {}
    for (i,j), d in mid_acc.items():
        cnt = d['count']
        bigram_feats[(i,j)] = {
            'count':      cnt,
            'area_ratio': total_area.get(i,0.0) / total_area.get(j,1.0),
            'avg_x':      d['sx']/cnt,
            'avg_y':      d['sy']/cnt,
        }

    # --- TRIGRAMS (fully connected triples) ---------------------------------
    tri_acc = defaultdict(lambda: {'count':0,'sx':0.0,'sy':0.0})
    nodes = list(range(len(plan['rooms'])))
    for u, v, w in combinations(nodes, 3):
        if G.has_edge(u,v) and G.has_edge(v,w) and G.has_edge(u,w):
            cu = ROOM_CODE[plan['rooms'][u]['type']]
            cv = ROOM_CODE[plan['rooms'][v]['type']]
            cw = ROOM_CODE[plan['rooms'][w]['type']]
            i, j, k = sorted((cu, cv, cw))
            xtri = (plan['rooms'][u]['x0'] +
                    plan['rooms'][v]['x0'] +
                    plan['rooms'][w]['x0']) / 3.0
            ytri = (plan['rooms'][u]['y0'] +
                    plan['rooms'][v]['y0'] +
                    plan['rooms'][w]['y0']) / 3.0
            acc = tri_acc[(i,j,k)]
            acc['count'] += 1
            acc['sx']    += xtri
            acc['sy']    += ytri

    trigram_feats = {}
    for (i,j,k), d in tri_acc.items():
        cnt = d['count']
        # two ratios: i/j and j/k
        r1 = total_area.get(i,0.0) / total_area.get(j,1.0)
        r2 = total_area.get(j,0.0) / total_area.get(k,1.0)
        trigram_feats[(i,j,k)] = {
            'count':         cnt,
            'area_ratio_ij': r1,
            'area_ratio_jk': r2,
            'avg_x':         d['sx']/cnt,
            'avg_y':         d['sy']/cnt,
        }

    return {
        'unigram_feats': unigram_feats,
        'bigram_feats':  bigram_feats,
        'trigram_feats': trigram_feats,
    }

In [ ]:
all_results = {}
for plan in list_of_plan_dicts:
    feats = extract_features(plan, meta_df)
    all_results[plan['filename']] = feats

In [ ]:
# with open("nnary_features.pkl", "wb") as f:
#     pickle.dump(all_results, f)

In [34]:
# 1) define the universe of codes & combinations
codes      = list(range(1, 9))                       # 1..8
bigrams    = list(combinations(codes, 2))            # all (i,j), i<j
trigrams   = list(combinations(codes, 3))            # all (i,j,k), i<j<k

# 2) build the list of column names we want
cols = []

# unigram columns
for i in codes:
    cols += [
        f"uni_{i}_count",
        f"uni_{i}_total_area",
        f"uni_{i}_avg_x",
        f"uni_{i}_avg_y",
    ]

# bigram columns
for i,j in bigrams:
    cols += [
        f"bi_{i}_{j}_count",
        f"bi_{i}_{j}_area_ratio",
        f"bi_{i}_{j}_avg_x",
        f"bi_{i}_{j}_avg_y",
    ]

# trigram columns
for i,j,k in trigrams:
    cols += [
        f"tri_{i}_{j}_{k}_count",
        f"tri_{i}_{j}_{k}_area_ratio_ij",
        f"tri_{i}_{j}_{k}_area_ratio_jk",
        f"tri_{i}_{j}_{k}_avg_x",
        f"tri_{i}_{j}_{k}_avg_y",
    ]

# 3) flatten each file's features into a single dict
rows = []
for fn, feats in all_results.items():
    row = {'filename': fn}
    # unigrams
    for i in codes:
        u = feats['unigram_feats'].get(i, {})
        row[f"uni_{i}_count"]      = u.get('count')
        row[f"uni_{i}_total_area"] = u.get('total_area')
        row[f"uni_{i}_avg_x"]       = u.get('avg_x')
        row[f"uni_{i}_avg_y"]       = u.get('avg_y')

    # bigrams
    for (i,j) in bigrams:
        b = feats['bigram_feats'].get((i,j), {})
        row[f"bi_{i}_{j}_count"]       = b.get('count')
        row[f"bi_{i}_{j}_area_ratio"]  = b.get('area_ratio')
        row[f"bi_{i}_{j}_avg_x"]       = b.get('avg_x')
        row[f"bi_{i}_{j}_avg_y"]       = b.get('avg_y')

    # trigrams
    for (i,j,k) in trigrams:
        t = feats['trigram_feats'].get((i,j,k), {})
        row[f"tri_{i}_{j}_{k}_count"]             = t.get('count')
        row[f"tri_{i}_{j}_{k}_area_ratio_ij"]     = t.get('area_ratio_ij')
        row[f"tri_{i}_{j}_{k}_area_ratio_jk"]     = t.get('area_ratio_jk')
        row[f"tri_{i}_{j}_{k}_avg_x"]             = t.get('avg_x')
        row[f"tri_{i}_{j}_{k}_avg_y"]             = t.get('avg_y')

    rows.append(row)

# 4) build the DataFrame
df = pd.DataFrame(rows).set_index('filename')[cols]

In [40]:
df

,uni_1_count,uni_1_total_area,uni_1_avg_x,uni_1_avg_y,uni_2_count,uni_2_total_area,uni_2_avg_x,uni_2_avg_y,uni_3_count,uni_3_total_area,...,tri_5_7_8_count,tri_5_7_8_area_ratio_ij,tri_5_7_8_area_ratio_jk,tri_5_7_8_avg_x,tri_5_7_8_avg_y,tri_6_7_8_count,tri_6_7_8_area_ratio_ij,tri_6_7_8_area_ratio_jk,tri_6_7_8_avg_x,tri_6_7_8_avg_y
filename,,,,,,,,,,,,,,,,,,,,,
40795.png,2.0,3396.0,-38.604240,53.561837,1.0,2542.0,72.000000,60.500000,1,6998.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
60259.png,NaN,NaN,NaN,NaN,1.0,6864.0,-46.000000,4.000000,1,3596.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
45409.png,2.0,4462.0,-2.314433,115.000000,1.0,2156.0,-31.000000,4.500000,1,5624.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
22733.png,1.0,1855.0,43.500000,80.500000,1.0,2560.0,87.000000,65.000000,1,6431.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25886.png,1.0,1591.0,72.500000,46.500000,1.0,2433.5,73.436340,99.513321,1,6373.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26753.png,1.0,2135.5,39.887224,53.269570,1.0,2359.5,18.394646,150.781027,1,6447.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
42387.png,1.0,2378.0,-76.000000,37.500000,1.0,2575.0,-101.982524,85.307702,1,8847.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19466.png,2.0,3132.0,49.873563,51.637931,NaN,NaN,NaN,NaN,1,6677.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [39]:
df.to_csv("nnary_features.csv")

In [ ]:
from collections import defaultdict
from itertools import combinations

import networkx as nx
import pandas as pd

ROOM_CODE = {
    'common_room': 1,
    'master_room': 2,
    'living_room': 3,
    'balcony': 4,
    'bathroom': 5,
    'kitchen': 6,
    'storage': 7,
    'dining': 8,
}


def recenter_and_normalize(rooms, center_x, center_y: float) -> None:
    """
    Mutates each room dict in-place, adding 'x0' and 'y0' keys for recentered coordinates.
    Y-axis is flipped so that positive Y points up.
    """
    for room in rooms:
        x, y = room['centroid']
        room['x0'] = x - center_x
        room['y0'] = -(y - center_y)


def accumulate_unigrams(
    rooms: List[Dict]
) -> Tuple[Dict[int, Dict], Dict[int, float]]:
    """
    Aggregates counts, area sums, and weighted sums of recentered coordinates by room code.
    Returns the raw accumulator and total area per code for later ratio calculations.
    """
    acc = defaultdict(lambda: {'count': 0, 'area': 0.0, 'sum_xa': 0.0, 'sum_ya': 0.0})
    for room in rooms:
        code = ROOM_CODE[room['type']]
        area = room['area']
        acc[code]['count'] += 1
        acc[code]['area'] += area
n        acc[code]['sum_xa'] += area * room['x0']
        acc[code]['sum_ya'] += area * room['y0']

    total_area = {code: data['area'] for code, data in acc.items()}
    return acc, total_area


def compute_unigram_features(
    acc: Dict[int, Dict]
) -> Dict[int, Dict[str, float]]:
    """
    Converts the raw unigram accumulator into normalized feature dict.
    """
    feats = {}
    for code, data in acc.items():
        area = data['area']
        feats[code] = {
            'count': data['count'],
            'total_area': area,
            'avg_x': data['sum_xa'] / area if area > 0 else 0.0,
            'avg_y': data['sum_ya'] / area if area > 0 else 0.0,
        }
    return feats


def compute_edge_midpoints(
    rooms: List[Dict], graph: nx.Graph
) -> Dict[Tuple[int, int], Dict[str, float]]:
    """
    For each edge in the graph, compute midpoint of the two room centroids.
    Accumulate counts and sum of midpoints keyed by sorted code pair.
    """
    acc = defaultdict(lambda: {'count': 0, 'sum_x': 0.0, 'sum_y': 0.0})
    for u, v in graph.edges():
        code_u = ROOM_CODE[rooms[u]['type']]
        code_v = ROOM_CODE[rooms[v]['type']]
        key = tuple(sorted((code_u, code_v)))
        x_mid = (rooms[u]['x0'] + rooms[v]['x0']) * 0.5
        y_mid = (rooms[u]['y0'] + rooms[v]['y0']) * 0.5
        acc[key]['count'] += 1
        acc[key]['sum_x'] += x_mid
        acc[key]['sum_y'] += y_mid
    return acc


def compute_bigram_features(
    mid_acc: Dict[Tuple[int, int], Dict[str, float]],
    total_area: Dict[int, float]
) -> Dict[Tuple[int, int], Dict[str, float]]:
    """
    Converts edge midpoint accumulators into feature dict with counts, area ratios, and avg midpoints.
    """
    feats = {}
    for (i, j), data in mid_acc.items():
        cnt = data['count']
        # avoid division by zero by defaulting denom to 1.0
        feats[(i, j)] = {
            'count': cnt,
            'area_ratio': total_area.get(i, 0.0) / total_area.get(j, 1.0),
            'avg_x': data['sum_x'] / cnt,
            'avg_y': data['sum_y'] / cnt,
        }
    return feats


def compute_trigram_features(
    rooms: List[Dict], graph: nx.Graph,
    total_area: Dict[int, float]
) -> Dict[Tuple[int, int, int], Dict[str, float]]:
    """
    Identifies fully connected triples (cliques of size 3), accumulates centroid averages and computes area ratios.
    """
    acc = defaultdict(lambda: {'count': 0, 'sum_x': 0.0, 'sum_y': 0.0})
    n = len(rooms)

    for u, v, w in combinations(range(n), 3):
        if graph.has_edge(u, v) and graph.has_edge(v, w) and graph.has_edge(u, w):
            codes = sorted((
                ROOM_CODE[rooms[u]['type']],
                ROOM_CODE[rooms[v]['type']],
                ROOM_CODE[rooms[w]['type']]
            ))
            x_avg = (rooms[u]['x0'] + rooms[v]['x0'] + rooms[w]['x0']) / 3.0
            y_avg = (rooms[u]['y0'] + rooms[v]['y0'] + rooms[w]['y0']) / 3.0
            acc[tuple(codes)]['count'] += 1
            acc[tuple(codes)]['sum_x'] += x_avg
            acc[tuple(codes)]['sum_y'] += y_avg

    feats = {}
    for (i, j, k), data in acc.items():
        cnt = data['count']
        feats[(i, j, k)] = {
            'count': cnt,
            'area_ratio_ij': total_area.get(i, 0.0) / total_area.get(j, 1.0),
            'area_ratio_jk': total_area.get(j, 0.0) / total_area.get(k, 1.0),
            'avg_x': data['sum_x'] / cnt,
            'avg_y': data['sum_y'] / cnt,
        }
    return feats


def extract_features(
    plan: Dict, meta_df: pd.DataFrame
) -> Dict[str, Dict]:
    """
    Main entry: extracts unigram, bigram, and trigram features from floorplan.

    plan: {
        'filename': str,
        'rooms': List[Dict],  # keys: 'type', 'centroid', 'area'
        'graph': nx.Graph  # edges between room indices
    }
    meta_df: DataFrame indexed by filename, with center_x, center_y
    """
    meta = meta_df.loc[plan['filename']]
    center_x, center_y = meta['center_x'], meta['center_y']

    rooms = plan['rooms']
    graph = plan['graph']

    # Recenter coordinates
    recenter_and_normalize(rooms, center_x, center_y)

    # Unigrams
    uni_acc, total_area = accumulate_unigrams(rooms)
    unigram_feats = compute_unigram_features(uni_acc)

    # Bigrams
    mid_acc = compute_edge_midpoints(rooms, graph)
    bigram_feats = compute_bigram_features(mid_acc, total_area)

    # Trigrams
    trigram_feats = compute_trigram_features(rooms, graph, total_area)

    return {
        'unigram_feats': unigram_feats,
        'bigram_feats': bigram_feats,
        'trigram_feats': trigram_feats,
    }


In [ ]:
def unigram_extractor(rooms):
    """
    Compute per-room stats.

    Parameters:
        rooms (list): each with 'type', 'area', 'x0', 'y0'
    Returns:
        feats (dict): code:{'count','total_area','avg_x','avg_y'}
        total_area (dict): code:total_area
    """
    stats = defaultdict(lambda: [0, 0.0, 0.0, 0.0])
    for room in rooms:
        code = ROOM_CODE[room['type']]
        area = room['area']
        x0, y0 = room['x0'], room['y0']
        count, area_sum, xsum, ysum = stats[code]

        stats[code] = [
            count + 1,
            area_sum + area,
            xsum + area * x0,
            ysum + area * y0
        ]

    feats = {}
    total_area_by_code = {}
    for code, (count, area_sum, xsum, ysum) in stats.items():
        total_area_by_code[code] = area_sum
        avg_x = xsum / area_sum if area_sum else 0.0
        avg_y = ysum / area_sum if area_sum else 0.0
        feats[code] = {
            'count': count,
            'total_area': area_sum,
            'avg_x': avg_x,
            'avg_y': avg_y
        }

    return feats, total_area_by_code

def bigram_extractor(rooms, graph, total_area_by_code):
    """
    Compute stats for adjacent room pairs.

    Parameters:
        rooms (list), graph (nx.Graph), total_area (dict)
    Returns:
        dict: (code_i,code_j):{'count','area_ratio','avg_x','avg_y'}
    """
    pair_stats = defaultdict(lambda: [0, 0.0, 0.0])

    for u, v in graph.edges():
        code_u = ROOM_CODE[rooms[u]['type']]
        code_v = ROOM_CODE[rooms[v]['type']]
        key = tuple(sorted((code_u, code_v)))

        x_mid = (rooms[u]['x0'] + rooms[v]['x0']) / 2.0
        y_mid = (rooms[u]['y0'] + rooms[v]['y0']) / 2.0

        count, xsum, ysum = pair_stats[key]
        pair_stats[key] = [count + 1, xsum + x_mid, ysum + y_mid]

    feats = {}
    for (i, j), (count, xsum, ysum) in pair_stats.items():
        avg_x = xsum / count
        avg_y = ysum / count
        # avoid division by zero
        area_i = total_area_by_code.get(i, 0.0)
        area_j = total_area_by_code.get(j, 1.0)
        area_ratio = area_i / area_j if area_j else 0.0

        feats[(i, j)] = {
            'count': count,
            'area_ratio': area_ratio,
            'avg_x': avg_x,
            'avg_y': avg_y
        }

    return feats

def trigram_extractor(rooms, graph, total_area_by_code):
    """
    Compute stats for room triangles.

    Parameters:
        rooms (list), graph (nx.Graph), total_area (dict)
    Returns:
        dict: (code_i,code_j,code_k):{'count','area_ratio_ij','area_ratio_jk','avg_x','avg_y'}
    """
    tri_stats = defaultdict(lambda: [0, 0.0, 0.0])
    n_rooms = len(rooms)

    for u, v, w in combinations(range(n_rooms), 3):
        if graph.has_edge(u, v) and graph.has_edge(v, w) and graph.has_edge(u, w):
            codes = sorted((
                ROOM_CODE[rooms[u]['type']],
                ROOM_CODE[rooms[v]['type']],
                ROOM_CODE[rooms[w]['type']]
            ))
            key = tuple(codes)

            x_avg = (rooms[u]['x0'] + rooms[v]['x0'] + rooms[w]['x0']) / 3.0
            y_avg = (rooms[u]['y0'] + rooms[v]['y0'] + rooms[w]['y0']) / 3.0

            count, xsum, ysum = tri_stats[key]
            tri_stats[key] = [count + 1, xsum + x_avg, ysum + y_avg]


    feats = {}
    for (i, j, k), (count, xsum, ysum) in tri_stats.items():
        avg_x = xsum / count
        avg_y = ysum / count

        # avoid division by zero
        area_i = total_area_by_code.get(i, 0.0)
        area_j = total_area_by_code.get(j, 1.0)
        area_k = total_area_by_code.get(k, 1.0)

        ratio_ij = area_i / area_j if area_j else 0.0
        ratio_jk = area_j / area_k if area_k else 0.0

        feats[(i, j, k)] = {
            'count': count,
            'area_ratio_ij': ratio_ij,
            'area_ratio_jk': ratio_jk,
            'avg_x': avg_x,
            'avg_y': avg_y
        }

    return feats

def extract_features(plan, meta_df):
    """
    Extract unigram, bigram, and trigram features from a floorplan.

    Parameters:
        plan (dict): {
            'filename': str,
            'rooms': list of {'type', 'centroid', 'area'},
            'graph': networkx.Graph of room adjacencies
        }
        meta_df (DataFrame): index: filename, cols:'center_x', 'center_y'

    Returns:
        dict: {
            'unigram_feats', 'bigram_feats', 'trigram_feats'
        }
    """

    row = meta_df.loc[plan['filename']]
    cx, cy = row['center_x'], row['center_y']
    rooms, G = plan['rooms'], plan['graph']

    # recenter coords
    for r in rooms:
        x,y = r['centroid']; r['x0'], r['y0'] = x-cx, -(y-cy)

    # compute n-gram features
    uni_feats, total_area = unigram_extractor(rooms)
    bi_feats = bigram_extractor(rooms, G, total_area)
    tri_feats = trigram_extractor(rooms, G, total_area)

    return {'unigram_feats': uni_feats,
            'bigram_feats':  bi_feats,
            'trigram_feats': tri_feats}


In [43]:
all_results_new = {}
for plan in list_of_plan_dicts:
    feats = extract_features(plan, meta_df)
    all_results_new[plan['filename']] = feats

In [ ]:
all_results_new

{'40795.png': {'unigram_feats': {1: {'count': 2,
    'total_area': 3396.0,
    'avg_x': -38.60424028268551,
    'avg_y': 53.561837455830386},
   2: {'count': 1, 'total_area': 2542.0, 'avg_x': 72.0, 'avg_y': 60.5},
   3: {'count': 1,
    'total_area': 6998.5,
    'avg_x': 35.88495153723417,
    'avg_y': 15.04617656164416},
   4: {'count': 1, 'total_area': 615.0, 'avg_x': 114.5, 'avg_y': 60.5},
   5: {'count': 1, 'total_area': 621.0, 'avg_x': 23.5, 'avg_y': 69.5},
   6: {'count': 1, 'total_area': 882.0, 'avg_x': -67.0, 'avg_y': 12.5}},
  'bigram_feats': {(1, 1): {'count': 1,
    'area_ratio': 1.0,
    'avg_x': -42.5,
    'avg_y': 52.25},
   (1, 3): {'count': 2,
    'area_ratio': 0.4852468386082732,
    'avg_x': -3.3075242313829136,
    'avg_y': 33.64808828082208},
   (1, 5): {'count': 1,
    'area_ratio': 5.468599033816425,
    'avg_x': 2.75,
    'avg_y': 65.0},
   (1, 6): {'count': 1,
    'area_ratio': 3.8503401360544216,
    'avg_x': -67.0,
    'avg_y': 28.25},
   (2, 3): {'count': 1,


: 